<a href="https://colab.research.google.com/github/qiujunzhang03-7/5243_Project3/blob/main/EDA%2BVisualization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
# Simulate raw event log data close to the real Travel Planner app
# Output columns:
# session_id, group, event, step, extra, timestamp

set.seed(123)

library(dplyr)
library(purrr)
library(stringr)
library(tidyr)

# 1. Basic settings
n_sessions <- 220

new_sid <- function() {
  paste0(
    format(Sys.time(), "%Y%m%d%H%M%S"),
    "_",
    paste0(sample(c(letters, 0:9), 6, replace = TRUE), collapse = "")
  )
}

simulate_one_session <- function(session_id,
                                 group,
                                 start_time,
                                 completion_prob_A = 0.58,
                                 completion_prob_B = 0.72,
                                 back_prob_A = 0.22,
                                 back_prob_B = 0.18,
                                 dup_start_prob = 0.45) {

  # completion
  if (group == "A") {
    completed <- rbinom(1, 1, completion_prob_A)
    used_back <- rbinom(1, 1, back_prob_A)
  } else {
    completed <- rbinom(1, 1, completion_prob_B)
    used_back <- rbinom(1, 1, back_prob_B)
  }

  n_back <- if (used_back == 1) sample(1:3, 1) else 0
  dup_start <- if (used_back == 1) rbinom(1, 1, dup_start_prob) else 0

  # app logic: steps 1–7, completed at step 8
  if (completed == 1) {
    final_step <- 8
    dropout_step <- NA_integer_
  } else {
    if (group == "A") {
      dropout_step <- sample(2:6, 1, prob = c(0.15, 0.25, 0.25, 0.20, 0.15))
    } else {
      dropout_step <- sample(3:7, 1, prob = c(0.10, 0.20, 0.25, 0.25, 0.20))
    }
    final_step <- dropout_step
  }

  current_time <- start_time
  out <- list()

  add_row <- function(event_name, step_val, extra_val = "") {
    tibble(
      session_id = session_id,
      group = group,
      event = event_name,
      step = as.integer(step_val),
      extra = extra_val,
      timestamp = format(current_time, "%Y-%m-%d %H:%M:%OS3")
    )
  }

  # session start
  out[[length(out) + 1]] <- add_row("session_start", 0, "returning=FALSE")
  current_time <- current_time + runif(1, 0.5, 2)

  # duplicate false session_start bug
  if (dup_start == 1) {
    out[[length(out) + 1]] <- add_row(
      "session_start",
      0,
      "returning=FALSE;bug_duplicate=TRUE"
    )
    current_time <- current_time + runif(1, 0.5, 2)
  }

  # step_next events
  if (completed == 1) {
    next_steps <- 1:7
  } else {
    if (dropout_step > 1) {
      next_steps <- 1:(dropout_step - 1)
    } else {
      next_steps <- integer(0)
    }
  }

  if (length(next_steps) > 0) {
    for (s in next_steps) {
      secs_here <- round(runif(1, 5, 30), 1)
      out[[length(out) + 1]] <- add_row(
        "step_next",
        s,
        paste0("time_secs=", secs_here)
      )
      current_time <- current_time + secs_here
    }
  }

  # step_back events
  if (n_back > 0) {
    if (completed == 1) {
      max_back_step <- 7
    } else {
      max_back_step <- max(2, final_step)
    }

    possible_back_steps <- 2:max_back_step

    for (i in seq_len(n_back)) {
      back_step <- sample(possible_back_steps, 1)
      secs_here <- round(runif(1, 3, 20), 1)
      out[[length(out) + 1]] <- add_row(
        "step_back",
        back_step,
        paste0("time_secs=", secs_here)
      )
      current_time <- current_time + secs_here
    }
  }

  # final events
  if (completed == 1) {
    out[[length(out) + 1]] <- add_row("completed", 8, "")
    current_time <- current_time + runif(1, 1, 5)

    last_step_secs <- round(runif(1, 5, 30), 1)
    out[[length(out) + 1]] <- add_row(
      "session_end",
      8,
      paste0("completed=TRUE;last_step_secs=", last_step_secs)
    )
  } else {
    secs_last <- round(runif(1, 5, 25), 1)
    out[[length(out) + 1]] <- add_row(
      "dropout",
      dropout_step,
      paste0("dropout_at_step=", dropout_step, ";secs_on_step=", secs_last)
    )
    current_time <- current_time + secs_last

    last_step_secs <- round(runif(1, 5, 30), 1)
    out[[length(out) + 1]] <- add_row(
      "session_end",
      dropout_step,
      paste0("completed=FALSE;last_step_secs=", last_step_secs)
    )
  }

  bind_rows(out) %>%
    arrange(timestamp)
}

# 2. Generate all raw event logs
start_base <- Sys.time() - 14 * 24 * 3600

events_df <- map_dfr(1:n_sessions, function(i) {
  session_id <- new_sid()
  group <- sample(c("A", "B"), 1)
  start_time <- start_base + runif(1, 0, 14 * 24 * 3600)

  simulate_one_session(
    session_id = session_id,
    group = group,
    start_time = start_time
  )
})

events_df <- events_df %>%
  arrange(timestamp, session_id)

# preview
print(head(events_df, 20))
print(table(events_df$event, events_df$group))

# 3. Optional session summary
session_summary <- events_df %>%
  group_by(session_id, group) %>%
  summarise(
    n_session_start = sum(event == "session_start"),
    has_completed = any(event == "completed"),
    dropout_step = ifelse(any(event == "dropout"),
                          step[event == "dropout"][1],
                          NA_integer_),
    n_step_next = sum(event == "step_next"),
    n_step_back = sum(event == "step_back"),
    .groups = "drop"
  ) %>%
  mutate(
    completed_task = ifelse(has_completed, 1, 0),
    steps_completed = ifelse(completed_task == 1, 8, dropout_step)
  )

print(head(session_summary, 10))

# 4. Save files
write.csv(events_df, "simulated_raw_events_like_sheet.csv", row.names = FALSE)
write.csv(session_summary, "simulated_session_summary_from_events.csv", row.names = FALSE)

# A tibble: 20 × 6
   session_id            group event          step extra               timestamp
   <chr>                 <chr> <chr>         <int> <chr>               <chr>    
 1 20260418053101_1igxrx B     session_start     0 "returning=FALSE"   2026-04-…
 2 20260418053101_1igxrx B     step_next         1 "time_secs=25.5"    2026-04-…
 3 20260418053101_1igxrx B     step_next         2 "time_secs=7.9"     2026-04-…
 4 20260418053101_1igxrx B     step_next         3 "time_secs=21.6"    2026-04-…
 5 20260418053101_1igxrx B     step_next         4 "time_secs=10"      2026-04-…
 6 20260418053101_1igxrx B     step_next         5 "time_secs=19.7"    2026-04-…
 7 20260418053101_1igxrx B     step_next         6 "time_secs=22.4"    2026-04-…
 8 20260418053101_1igxrx B     step_next         7 "time_secs=18.2"    2026-04-…
 9 20260418053101_1igxrx B     completed         8 ""                  2026-04-…
10 20260418053101_1igxrx B     session_end       8 "completed=TRUE;la… 2026-04-…
11 202604

In [15]:
install.packages(c("readxl", "dplyr", "stringr", "ggplot2", "tidyr"))

Installing packages into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



In [16]:
# CLEANING CODE
# Outputs:
# 1. CLEAN_real_sessions.csv
# 2. CLEAN_final_combined_dataset.csv
# 3. CLEAN_cleaning_summary.csv

library(readxl)
library(dplyr)
library(stringr)

# 1. Read raw data
real_events <- read_excel("travel_planner_ab_test.xlsx", sheet = "events")
sim_summary <- read.csv("simulated_session_summary_from_events.csv", stringsAsFactors = FALSE)

real_events <- real_events %>%
  mutate(
    timestamp = as.POSIXct(timestamp, format = "%Y-%m-%d %H:%M:%OS", tz = "UTC"),
    session_id = as.character(session_id),
    group = as.character(group),
    event = as.character(event),
    extra = as.character(extra)
  )

# 2. Basic row-level filtering
valid_events <- c(
  "session_start", "step_next", "step_back",
  "completed", "restart", "dropout", "session_end"
)

raw_n_rows <- nrow(real_events)
raw_n_sessions <- n_distinct(real_events$session_id)

events_kept <- real_events %>%
  filter(!is.na(session_id), session_id != "") %>%
  filter(group %in% c("A", "B")) %>%
  filter(event %in% valid_events) %>%
  arrange(session_id, timestamp)

after_basic_rows <- nrow(events_kept)
after_basic_sessions <- n_distinct(events_kept$session_id)

# 3. Session-level flags
session_flags <- events_kept %>%
  group_by(session_id) %>%
  summarise(
    group_mode = names(sort(table(group), decreasing = TRUE))[1],

    n_rows = n(),
    n_session_start = sum(event == "session_start", na.rm = TRUE),
    n_completed = sum(event == "completed", na.rm = TRUE),
    n_dropout = sum(event == "dropout", na.rm = TRUE),
    n_session_end = sum(event == "session_end", na.rm = TRUE),
    n_restart = sum(event == "restart", na.rm = TRUE),
    n_step_next = sum(event == "step_next", na.rm = TRUE),
    n_step_back = sum(event == "step_back", na.rm = TRUE),

    has_completed = any(event == "completed", na.rm = TRUE),
    has_dropout = any(event == "dropout", na.rm = TRUE),
    has_session_end = any(event == "session_end", na.rm = TRUE),

    first_time = min(timestamp, na.rm = TRUE),
    last_time = max(timestamp, na.rm = TRUE),

    returning_flag = any(str_detect(extra[event == "session_start"], "returning=TRUE"), na.rm = TRUE),
    duplicate_bug_flag = any(str_detect(extra[event == "session_start"], "bug_duplicate=TRUE"), na.rm = TRUE),

    .groups = "drop"
  ) %>%
  mutate(
    session_duration_sec = as.numeric(difftime(last_time, first_time, units = "secs")),

    flag_duplicate_start = n_session_start > 1,
    flag_no_start = n_session_start == 0,
    flag_multiple_completed = n_completed > 1,
    flag_completed_and_dropout = has_completed & has_dropout,
    flag_too_few_events = n_rows <= 1,

    exclusion_reason = case_when(
      flag_no_start ~ "No session_start",
      flag_multiple_completed ~ "Multiple completed events",
      flag_completed_and_dropout ~ "Completed and dropout both present",
      flag_too_few_events ~ "Only one event row",
      TRUE ~ NA_character_
    ),

    exclude_from_real_analysis = !is.na(exclusion_reason)
  )

valid_session_ids <- session_flags %>%
  filter(!exclude_from_real_analysis) %>%
  pull(session_id)

# 4. Clean real sessions
clean_events <- events_kept %>%
  filter(session_id %in% valid_session_ids)

clean_real_sessions <- clean_events %>%
  group_by(session_id) %>%
  summarise(
    group = names(sort(table(group), decreasing = TRUE))[1],

    n_session_start = sum(event == "session_start", na.rm = TRUE),
    n_step_next = sum(event == "step_next", na.rm = TRUE),
    n_step_back = sum(event == "step_back", na.rm = TRUE),
    n_restart = sum(event == "restart", na.rm = TRUE),

    has_completed = any(event == "completed", na.rm = TRUE),
    has_dropout = any(event == "dropout", na.rm = TRUE),

    dropout_step = ifelse(any(event == "dropout"),
                          as.numeric(step[event == "dropout"][1]),
                          NA_real_),

    returning = any(str_detect(extra[event == "session_start"], "returning=TRUE"), na.rm = TRUE),
    duplicate_session_start = any(str_detect(extra[event == "session_start"], "bug_duplicate=TRUE"), na.rm = TRUE) |
      sum(event == "session_start", na.rm = TRUE) > 1,

    first_time = min(timestamp, na.rm = TRUE),
    last_time = max(timestamp, na.rm = TRUE),

    .groups = "drop"
  ) %>%
  mutate(
    completed_task = ifelse(has_completed, 1, 0),
    steps_completed = ifelse(completed_task == 1, 8, dropout_step),
    session_duration_sec = as.numeric(difftime(last_time, first_time, units = "secs")),
    source = "real"
  )

write.csv(clean_real_sessions, "CLEAN_real_sessions.csv", row.names = FALSE)

# 5. Final combined dataset
#    Keep all real sessions, supplement with simulated data
sim_summary <- sim_summary %>%
  mutate(source = "simulated")

target_per_group <- 60

real_A <- clean_real_sessions %>% filter(group == "A")
real_B <- clean_real_sessions %>% filter(group == "B")

need_A <- max(target_per_group - nrow(real_A), 0)
need_B <- max(target_per_group - nrow(real_B), 0)

sim_A <- sim_summary %>% filter(group == "A") %>% slice_head(n = need_A)
sim_B <- sim_summary %>% filter(group == "B") %>% slice_head(n = need_B)

final_combined <- bind_rows(clean_real_sessions, sim_A, sim_B)

write.csv(final_combined, "CLEAN_final_combined_dataset.csv", row.names = FALSE)

# 6. Cleaning summary
cleaning_summary <- data.frame(
  metric = c(
    "Raw event rows",
    "Raw unique sessions",
    "Rows after basic filtering",
    "Sessions after basic filtering",
    "Sessions excluded from real analysis",
    "Valid real sessions retained",
    "Valid real A sessions",
    "Valid real B sessions",
    "Real sessions flagged duplicate_start",
    "Simulated A sessions added",
    "Simulated B sessions added",
    "Final combined A sessions",
    "Final combined B sessions",
    "Final combined total sessions"
  ),
  value = c(
    raw_n_rows,
    raw_n_sessions,
    after_basic_rows,
    after_basic_sessions,
    sum(session_flags$exclude_from_real_analysis, na.rm = TRUE),
    nrow(clean_real_sessions),
    nrow(real_A),
    nrow(real_B),
    sum(clean_real_sessions$duplicate_session_start, na.rm = TRUE),
    nrow(sim_A),
    nrow(sim_B),
    sum(final_combined$group == "A"),
    sum(final_combined$group == "B"),
    nrow(final_combined)
  )
)

write.csv(cleaning_summary, "CLEAN_cleaning_summary.csv", row.names = FALSE)

cat("Cleaning files saved:\n")
cat("- CLEAN_real_sessions.csv\n")
cat("- CLEAN_final_combined_dataset.csv\n")
cat("- CLEAN_cleaning_summary.csv\n")

Cleaning files saved:
- CLEAN_real_sessions.csv
- CLEAN_final_combined_dataset.csv
- CLEAN_cleaning_summary.csv


In [18]:
# EDA CODE
# Uses:
# - raw travel_planner_ab_test.xlsx
# - CLEAN_real_sessions.csv
# - simulated_session_summary_from_events.csv
library(readxl)
library(dplyr)
library(stringr)
library(ggplot2)

# 1. Read data
real_events <- read_excel("travel_planner_ab_test.xlsx", sheet = "events")
clean_real <- read.csv("CLEAN_real_sessions.csv", stringsAsFactors = FALSE)
sim_summary <- read.csv("simulated_session_summary_from_events.csv", stringsAsFactors = FALSE) %>%
  mutate(source = "simulated")

real_events <- real_events %>%
  mutate(
    timestamp = as.POSIXct(timestamp, format = "%Y-%m-%d %H:%M:%OS", tz = "UTC"),
    event = as.character(event),
    group = as.character(group),
    extra = as.character(extra)
  ) %>%
  filter(group %in% c("A", "B"))

# 2. EDA Figure 1: Sample size by source and group
eda_sample <- bind_rows(
  clean_real %>% select(session_id, group, source),
  sim_summary %>% select(session_id, group, source)
) %>%
  count(source, group)

p1 <- ggplot(eda_sample, aes(x = group, y = n, fill = source)) +
  geom_col(position = "dodge") +
  labs(
    title = "Sample Size by Source and Group",
    x = "Group",
    y = "Number of Sessions"
  ) +
  theme_minimal(base_size = 13)

ggsave("EDA_1_sample_size_by_source_group.png", p1, width = 7, height = 4, dpi = 300)

# 3. EDA Figure 2: Raw event type distribution
eda_event_dist <- real_events %>%
  count(group, event)

p2 <- ggplot(eda_event_dist, aes(x = event, y = n, fill = group)) +
  geom_col(position = "dodge") +
  labs(
    title = "Raw Event Type Distribution by Group",
    x = "Event Type",
    y = "Count"
  ) +
  theme_minimal(base_size = 13) +
  theme(axis.text.x = element_text(angle = 35, hjust = 1))

ggsave("EDA_2_event_type_distribution.png", p2, width = 8, height = 4.5, dpi = 300)

# 4. EDA Figure 3: Duplicate session_start diagnostic
eda_dup <- clean_real %>%
  mutate(dup_label = ifelse(duplicate_session_start, "Duplicate start", "Single start")) %>%
  count(group, dup_label)

p3 <- ggplot(eda_dup, aes(x = dup_label, y = n, fill = group)) +
  geom_col(position = "dodge") +
  labs(
    title = "Duplicate Session Start Diagnostic",
    x = "Session Start Status",
    y = "Number of Sessions"
  ) +
  theme_minimal(base_size = 13)

ggsave("EDA_3_duplicate_session_start.png", p3, width = 7, height = 4, dpi = 300)

# 5. EDA Figure 4: Raw dropout pattern in real data
eda_dropout <- clean_real %>%
  filter(completed_task == 0, !is.na(dropout_step)) %>%
  count(group, dropout_step)

p4 <- ggplot(eda_dropout, aes(x = dropout_step, y = n, color = group)) +
  geom_line(linewidth = 1) +
  geom_point(size = 2) +
  labs(
    title = "Raw Dropout Pattern in Real Data",
    x = "Dropout Step",
    y = "Number of Sessions"
  ) +
  theme_minimal(base_size = 13)

ggsave("EDA_4_raw_dropout_real_data.png", p4, width = 7, height = 4, dpi = 300)

# 6. EDA summary tables
eda_table_real <- clean_real %>%
  group_by(group) %>%
  summarise(
    sessions = n(),
    completion_rate = mean(completed_task, na.rm = TRUE),
    avg_steps_completed = mean(steps_completed, na.rm = TRUE),
    duplicate_start_rate = mean(duplicate_session_start, na.rm = TRUE),
    avg_back_clicks = mean(n_step_back, na.rm = TRUE),
    .groups = "drop"
  )

eda_table_source <- bind_rows(
  clean_real %>% select(session_id, group, completed_task, steps_completed, source),
  sim_summary %>% select(session_id, group, completed_task, steps_completed, source)
) %>%
  group_by(source, group) %>%
  summarise(
    sessions = n(),
    completion_rate = mean(completed_task, na.rm = TRUE),
    avg_steps_completed = mean(steps_completed, na.rm = TRUE),
    .groups = "drop"
  )

write.csv(eda_table_real, "EDA_table_real_summary.csv", row.names = FALSE)
write.csv(eda_table_source, "EDA_table_source_summary.csv", row.names = FALSE)

cat("EDA files saved:\n")
cat("- EDA_1_sample_size_by_source_group.png\n")
cat("- EDA_2_event_type_distribution.png\n")
cat("- EDA_3_duplicate_session_start.png\n")
cat("- EDA_4_raw_dropout_real_data.png\n")
cat("- EDA_table_real_summary.csv\n")
cat("- EDA_table_source_summary.csv\n")

`geom_line()`: Each group consists of only one observation.
ℹ Do you need to adjust the group aesthetic?


EDA files saved:
- EDA_1_sample_size_by_source_group.png
- EDA_2_event_type_distribution.png
- EDA_3_duplicate_session_start.png
- EDA_4_raw_dropout_real_data.png
- EDA_table_real_summary.csv
- EDA_table_source_summary.csv


In [ ]:
combined %>% count(group)

group,n
<chr>,<int>
A,60
B,60


In [ ]:
combined %>%
  group_by(group) %>%
  summarise(completion_rate = mean(completed_task))

group,completion_rate
<chr>,<dbl>
A,0.3000000
B,0.7666667


In [ ]:
combined %>%
  group_by(group) %>%
  summarise(avg_steps_completed = mean(steps_completed, na.rm = TRUE))

group,avg_steps_completed
<chr>,<dbl>
A,3.916667
B,7.116667


In [ ]:
combined %>%
  filter(completed_task == 0, !is.na(dropout_step)) %>%
  count(group, dropout_step)

group,dropout_step,n
<chr>,<dbl>,<int>
A,1,26
A,2,2
A,3,5
A,4,4
A,6,5
B,1,4
B,3,1
B,4,2
B,5,1


In [19]:
# FINAL ANALYSIS CODE
# Uses:
# - CLEAN_final_combined_dataset.csv

library(dplyr)
library(ggplot2)
library(tidyr)

# 1. Read final cleaned dataset
final_data <- read.csv("CLEAN_final_combined_dataset.csv", stringsAsFactors = FALSE)

# 2. Main summary with 95% CI
final_summary <- final_data %>%
  group_by(group) %>%
  summarise(
    sessions = n(),
    completed = sum(completed_task, na.rm = TRUE),
    completion_rate = mean(completed_task, na.rm = TRUE),
    avg_steps_completed = mean(steps_completed, na.rm = TRUE),
    median_dropout_step = median(dropout_step, na.rm = TRUE),
    avg_back_clicks = mean(n_step_back, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  mutate(
    se = sqrt(completion_rate * (1 - completion_rate) / sessions),
    ci_low = pmax(0, completion_rate - 1.96 * se),
    ci_high = pmin(1, completion_rate + 1.96 * se)
  )

print(final_summary)

# 3. Statistical tests
completion_table <- final_data %>%
  group_by(group) %>%
  summarise(
    completed = sum(completed_task, na.rm = TRUE),
    total = n(),
    .groups = "drop"
  )

prop_test_result <- prop.test(
  x = completion_table$completed,
  n = completion_table$total
)

ttest_steps <- t.test(steps_completed ~ group, data = final_data)
wilcox_steps <- wilcox.test(steps_completed ~ group, data = final_data)

rate_A <- final_summary$completion_rate[final_summary$group == "A"]
rate_B <- final_summary$completion_rate[final_summary$group == "B"]

absolute_lift <- rate_B - rate_A
relative_lift <- (rate_B - rate_A) / rate_A

results_table <- data.frame(
  outcome = c("Completion rate", "Steps completed (t-test)", "Steps completed (Wilcoxon)"),
  group_A = c(
    rate_A,
    mean(final_data$steps_completed[final_data$group == "A"], na.rm = TRUE),
    median(final_data$steps_completed[final_data$group == "A"], na.rm = TRUE)
  ),
  group_B = c(
    rate_B,
    mean(final_data$steps_completed[final_data$group == "B"], na.rm = TRUE),
    median(final_data$steps_completed[final_data$group == "B"], na.rm = TRUE)
  ),
  statistic = c(
    unname(prop_test_result$statistic),
    unname(ttest_steps$statistic),
    unname(wilcox_steps$statistic)
  ),
  p_value = c(
    prop_test_result$p.value,
    ttest_steps$p.value,
    wilcox_steps$p.value
  )
)

effect_size_table <- data.frame(
  metric = c("Absolute lift in completion rate", "Relative lift in completion rate"),
  value = c(absolute_lift, relative_lift)
)

write.csv(results_table, "FINAL_statistical_results_table.csv", row.names = FALSE)
write.csv(effect_size_table, "FINAL_effect_size_table.csv", row.names = FALSE)

# 4. Final Figure 1: Completion rate with 95% CI
p1 <- ggplot(final_summary, aes(x = group, y = completion_rate)) +
  geom_col(width = 0.6) +
  geom_errorbar(aes(ymin = ci_low, ymax = ci_high), width = 0.15, linewidth = 0.8) +
  ylim(0, 1) +
  labs(
    title = "Completion Rate by Group with 95% Confidence Intervals",
    x = "Group",
    y = "Completion Rate"
  ) +
  theme_minimal(base_size = 13)

ggsave("FINAL_1_completion_rate_with_CI.png", p1, width = 7, height = 4.5, dpi = 300)

# 5. Final Figure 2: Steps completed
p2 <- ggplot(final_data, aes(x = group, y = steps_completed)) +
  geom_boxplot(outlier.shape = NA, width = 0.45) +
  geom_jitter(width = 0.12, alpha = 0.55, size = 1.8) +
  labs(
    title = "Steps Completed by Group",
    x = "Group",
    y = "Steps Completed"
  ) +
  theme_minimal(base_size = 13)

ggsave("FINAL_2_steps_completed_boxplot_jitter.png", p2, width = 7, height = 4.5, dpi = 300)

# 6. Final Figure 3: Retention curve
steps <- 1:8

retention_df <- expand.grid(
  group = c("A", "B"),
  step = steps
) %>%
  left_join(
    final_data %>%
      group_by(group) %>%
      summarise(total_sessions = n(), .groups = "drop"),
    by = "group"
  ) %>%
  rowwise() %>%
  mutate(
    retained_sessions = sum(final_data$group == group & final_data$steps_completed >= step, na.rm = TRUE),
    retained_rate = retained_sessions / total_sessions
  ) %>%
  ungroup()

p3 <- ggplot(retention_df, aes(x = step, y = retained_rate, color = group)) +
  geom_line(linewidth = 1.1) +
  geom_point(size = 2.2) +
  ylim(0, 1) +
  scale_x_continuous(breaks = 1:8) +
  labs(
    title = "Retention Across Steps by Group",
    x = "Step",
    y = "Proportion of Sessions Remaining"
  ) +
  theme_minimal(base_size = 13)

ggsave("FINAL_3_retention_curve.png", p3, width = 7, height = 4.5, dpi = 300)

# 7. Optional final sample-size plot
p4 <- ggplot(final_summary, aes(x = group, y = sessions)) +
  geom_col(width = 0.6) +
  labs(
    title = "Final Balanced Sample Size by Group",
    x = "Group",
    y = "Number of Sessions"
  ) +
  theme_minimal(base_size = 13)

ggsave("FINAL_4_balanced_sessions.png", p4, width = 6.5, height = 4, dpi = 300)

# 8. Save summary tables
write.csv(final_summary, "FINAL_summary_by_group.csv", row.names = FALSE)
write.csv(retention_df, "FINAL_retention_curve_data.csv", row.names = FALSE)

cat("Final files saved:\n")
cat("- FINAL_1_completion_rate_with_CI.png\n")
cat("- FINAL_2_steps_completed_boxplot_jitter.png\n")
cat("- FINAL_3_retention_curve.png\n")
cat("- FINAL_4_balanced_sessions.png\n")
cat("- FINAL_summary_by_group.csv\n")
cat("- FINAL_retention_curve_data.csv\n")
cat("- FINAL_statistical_results_table.csv\n")
cat("- FINAL_effect_size_table.csv\n")

# A tibble: 2 × 10
  group sessions completed completion_rate avg_steps_completed
  <chr>    <int>     <int>           <dbl>               <dbl>
1 A           60        18           0.3                  3.95
2 B           60        47           0.783                7.15
# ℹ 5 more variables: median_dropout_step <dbl>, avg_back_clicks <dbl>,
#   se <dbl>, ci_low <dbl>, ci_high <dbl>
Final files saved:
- FINAL_1_completion_rate_with_CI.png
- FINAL_2_steps_completed_boxplot_jitter.png
- FINAL_3_retention_curve.png
- FINAL_4_balanced_sessions.png
- FINAL_summary_by_group.csv
- FINAL_retention_curve_data.csv
- FINAL_statistical_results_table.csv
- FINAL_effect_size_table.csv
